In [ ]:
!pip install hmmlearn

In [ ]:
import numpy as np
import joblib
from hmmlearn import hmm
from sklearn.preprocessing import LabelEncoder
import pandas as pd

In [ ]:
# Check the columns in your CSV file
data1 = pd.read_csv('IDAN1.csv')
print(data1.columns)

In [ ]:
# Check the columns in your CSV file
data2 = pd.read_csv('IDAN2.csv')
print(data2.columns)

In [ ]:
# Check the columns in your CSV file
data3 = pd.read_csv('IDAN3.csv')
print(data3.columns)

In [ ]:
import pandas as pd

# Extract the 'Opcode' sequences from each dataset
opcode_sequences1 = data1['Opcode']
opcode_sequences2 = data2['Opcode']
opcode_sequences3 = data3['Opcode']

# If there are labels, extract them, otherwise initialize labels as None for now
labels1 = data1.get('label', None)
labels2 = data2.get('label', None)
labels3 = data3.get('label', None)

# Convert each sequence into a list of lists of individual opcodes
opcode_sequences1 = [sequence.split() for sequence in opcode_sequences1]
opcode_sequences2 = [sequence.split() for sequence in opcode_sequences2]
opcode_sequences3 = [sequence.split() for sequence in opcode_sequences3]

# Optionally, merge all datasets if needed
# For example, merging opcode sequences and labels from all datasets into one:
combined_opcode_sequences = opcode_sequences1 + opcode_sequences2 + opcode_sequences3
combined_labels = None
if labels1 is not None and labels2 is not None and labels3 is not None:
    combined_labels = pd.concat([labels1, labels2, labels3])

# Display the first few sequences for inspection
print("\nCombined Opcode Sequences Sample:")
print(combined_opcode_sequences[:5])

# Check if labels are available, otherwise proceed with only sequences
if combined_labels is not None:
    print("\nCombined Labels Sample:")
    print(combined_labels.head())
else:
    print("\nNo labels found; proceeding with opcode sequences only.")


In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Flatten the combined opcode sequences to get a list of unique opcodes
flat_opcode_sequences = [opcode for sequence in combined_opcode_sequences for opcode in sequence]

# Use LabelEncoder to transform opcodes into numerical values
label_encoder = LabelEncoder()
label_encoder.fit(flat_opcode_sequences)

# Transform each opcode sequence into a sequence of integers
encoded_opcode_sequences = [label_encoder.transform(sequence) for sequence in combined_opcode_sequences]

# For training, HMM expects all sequences to be concatenated into a single long sequence
# and lengths array that indicates the number of opcodes in each sequence
concatenated_sequences = np.concatenate(encoded_opcode_sequences)
sequence_lengths = [len(sequence) for sequence in encoded_opcode_sequences]

# Check the results
print("\nEncoded Sequences Sample:")
print(encoded_opcode_sequences[:2])


In [ ]:
# Concatenate sequences and store their lengths
concatenated_sequences = np.concatenate(encoded_opcode_sequences)
sequence_lengths = [len(sequence) for sequence in encoded_opcode_sequences]

print(f"Concatenated Sequences: {concatenated_sequences[:20]}")
print(f"Sequence Lengths: {sequence_lengths[:10]}")


In [ ]:
# Add a small constant to ensure no zero rows in transmat_
def smooth_transmat(transmat, epsilon=1e-5):
    smoothed_transmat = transmat + epsilon
    smoothed_transmat = smoothed_transmat / smoothed_transmat.sum(axis=1, keepdims=True)
    return smoothed_transmat


In [ ]:
import numpy as np
from hmmlearn import hmm

# Define the number of hidden states for the HMM (e.g., 2 for malware and legit)
n_components = 2

# Initialize the HMM with proper parameters
model = hmm.MultinomialHMM(n_components=n_components, n_iter=100, random_state=42)

# Manually set the start probabilities and transition matrix
model.startprob_ = np.array([0.5, 0.5])  # Example: start with equal probabilities for both states
model.transmat_ = np.array([[0.7, 0.3],   # Example: transition probabilities from state 0
                            [0.3, 0.7]])  # Example: transition probabilities from state 1

# Train the HMM on the opcode sequences
model.fit(concatenated_sequences.reshape(-1, 1), sequence_lengths)

print("HMM Model Training Complete")

# Reinitialize transmat_ for rows that sum to zero
def reinitialize_transmat(transmat, epsilon=1e-5):
    for i in range(transmat.shape[0]):
        if transmat[i].sum() == 0:
            transmat[i] = np.full(transmat.shape[1], 1.0 / transmat.shape[1])
    return transmat

# After training the HMM
model.fit(concatenated_sequences.reshape(-1, 1), sequence_lengths)

# Apply smoothing and reinitialize zero-sum rows in transmat_
model.transmat_ = smooth_transmat(model.transmat_)
model.transmat_ = reinitialize_transmat(model.transmat_)

print("Reinitialized and Smoothed transition matrix:")
print(model.transmat_)

# Check and reinitialize startprob_ if it contains NaN
if np.isnan(model.startprob_).any():
    model.startprob_ = np.full(n_components, 1.0 / n_components)

# Verify startprob_ sums to 1
if not np.isclose(model.startprob_.sum(), 1.0):
    raise ValueError(f"Error: startprob_ must sum to 1 (got {model.startprob_.sum()})")

# Check if the transition matrix is valid
def check_transmat(model):
    try:
        model._check()
        print("Transition matrix is valid.")
    except ValueError as e:
        print(f"Error: {e}")

# After training the model
check_transmat(model)

In [ ]:
# Function to classify a new opcode sequence
def classify_opcode_sequence(opcode_sequence, trained_model, label_encoder):
    try:
        # Encode the sequence using the same label encoder
        encoded_sequence = label_encoder.transform(opcode_sequence)

        # Reshape to match the model input
        reshaped_sequence = np.array(encoded_sequence).reshape(-1, 1)

        # Compute the log likelihood for this sequence
        log_likelihood = trained_model.score(reshaped_sequence)

        # Based on log likelihood, classify as malware or legit
        if log_likelihood < -50:  # Adjust threshold based on your model's performance
            return "Malware"
        else:
            return "Legit"
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Example usage
new_opcode_sequence = ["mov", "add", "jmp", "push"]  # Example sequence
prediction = classify_opcode_sequence(new_opcode_sequence, model, label_encoder)
print(f"\nPrediction for new sequence: {prediction}")

In [1]:
!pip install hmmlearn

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from hmmlearn import hmm
import joblib

In [9]:
# Load datasets
data1 = pd.read_csv('IDAN1.csv')
data2 = pd.read_csv('IDAN2.csv')
data3 = pd.read_csv('IDAN3.csv')

# Check columns in each dataset
print("Columns in data1:", data1.columns)
print("Columns in data2:", data2.columns)
print("Columns in data3:", data3.columns)


Columns in data1: Index(['Opcode', 'Operand', 'Comment'], dtype='object')
Columns in data2: Index(['Opcode', 'Operand', 'Comment'], dtype='object')
Columns in data3: Index(['Opcode', 'Operand', 'Comment'], dtype='object')


In [10]:
# Extract opcode sequences from each dataset
opcode_sequences1 = data1['Opcode'].astype(str)
opcode_sequences2 = data2['Opcode'].astype(str)
opcode_sequences3 = data3['Opcode'].astype(str)

# Convert each sequence into a list of opcodes
opcode_sequences1 = [sequence.split() for sequence in opcode_sequences1]
opcode_sequences2 = [sequence.split() for sequence in opcode_sequences2]
opcode_sequences3 = [sequence.split() for sequence in opcode_sequences3]

# Combine all opcode sequences into one list
combined_opcode_sequences = opcode_sequences1 + opcode_sequences2 + opcode_sequences3

# Create a LabelEncoder to encode the opcodes
label_encoder = LabelEncoder()

# Flatten the list of lists to create a single list of all opcodes
all_opcodes = [opcode for sequence in combined_opcode_sequences for opcode in sequence]
label_encoder.fit(all_opcodes)

# Encode each sequence using the label encoder
encoded_opcode_sequences = [label_encoder.transform(sequence) for sequence in combined_opcode_sequences]

# Display a sample of the encoded sequences
print("Encoded Opcode Sequences Sample:", encoded_opcode_sequences[:3])


Encoded Opcode Sequences Sample: [array([236]), array([0]), array([8])]


In [11]:
# Concatenate all encoded sequences into a single array
concatenated_sequences = np.concatenate(encoded_opcode_sequences)

# Store the lengths of each sequence
sequence_lengths = [len(sequence) for sequence in encoded_opcode_sequences]

# Display concatenated sequences and their lengths
print("Concatenated Sequences:", concatenated_sequences[:20])
print("Sequence Lengths:", sequence_lengths[:10])


Concatenated Sequences: [236   0   8   8 246 258   9 142 138 143 298 258 262   5 243   5 247 244
 237  12]
Sequence Lengths: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [12]:
# Define number of hidden states for HMM (e.g., 2 for malware and legit)
n_components = 2

# Initialize HMM with specified parameters
model = hmm.MultinomialHMM(n_components=n_components, n_iter=100, random_state=42)

# Set initial start probabilities and transition matrix
model.startprob_ = np.array([0.5, 0.5])  # Equal probability for both states initially
model.transmat_ = np.array([
    [0.7, 0.3],  # Transition probabilities from state 0
    [0.3, 0.7]   # Transition probabilities from state 1
])


MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


In [13]:
# Train HMM model on encoded opcode sequences
model.fit(concatenated_sequences.reshape(-1, 1), sequence_lengths)

# Reinitialize transmat_ for rows that sum to zero
def reinitialize_transmat(transmat, epsilon=1e-5):
    for i in range(transmat.shape[0]):
        if transmat[i].sum() == 0:
            transmat[i] = np.full(transmat.shape[1], 1.0 / transmat.shape[1])
    return transmat

# Apply smoothing and reinitialize zero-sum rows in transmat_
model.transmat_ = reinitialize_transmat(model.transmat_)


Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Some rows of transmat_ have zero sum because no transition from the state was ever observed.


In [14]:
# Check and reinitialize startprob_ if it contains NaN
if np.isnan(model.startprob_).any():
    model.startprob_ = np.full(n_components, 1.0 / n_components)

# Verify startprob_ sums to 1
if not np.isclose(model.startprob_.sum(), 1.0):
    raise ValueError(f"Error: startprob_ must sum to 1 (got {model.startprob_.sum()})")

# Check if the transition matrix is valid
def check_transmat(model):
    try:
        model._check()
        print("Transition matrix is valid.")
    except ValueError as e:
        print(f"Error: {e}")

# After training the model
check_transmat(model)


Transition matrix is valid.


In [15]:
def classify_opcode_sequence(opcode_sequence, trained_model, label_encoder):
    try:
        # Encode the sequence using the same label encoder
        encoded_sequence = label_encoder.transform(opcode_sequence)

        # Reshape to match the model input
        reshaped_sequence = np.array(encoded_sequence).reshape(-1, 1)

        # Compute the log likelihood for this sequence
        log_likelihood = trained_model.score(reshaped_sequence)

        # Based on log likelihood, classify as malware or legit
        if log_likelihood < -50:  # Adjust threshold based on your model's performance
            return "Malware"
        else:
            return "Legit"
    except Exception as e:
        return f"Error: {str(e)}"

# Example usage
new_opcode_sequence = ["mov", "add", "jmp", "push"]  # Example sequence
prediction = classify_opcode_sequence(new_opcode_sequence, model, label_encoder)
print(f"\nPrediction for new sequence: {prediction}")



Prediction for new sequence: Legit
